# Semana 06 — Limpeza e Transformação de Dados

**Curso:** Análise de Dados com Python — SENAI (Turma T5)
**UC (MSEP):** Manipulação de Dados com Python e SQL (150h) — Bloco 3, Semanas 05 e 06 (fecha o bloco)

Na Semana 05 você explorou um dataset já "pronto". Esta semana ensina a deixar um dataset real — com nulos, duplicatas e outliers — pronto para análise.

> Em cada tópico abaixo: **2 exemplos resolvidos** + **1 atividade prática** para você fazer sozinho(a).

---
### 🟢 Abertura — Semana 06: Limpeza e Transformação de Dados

Dados reais nunca chegam perfeitos. <strong>Hoje você aprende a detectar e corrigir os problemas mais comuns</strong> antes de analisar.

**O que você vai aprender hoje:**
- Identificar e tratar valores ausentes (NaN) com estratégias adequadas
- Detectar e remover duplicidades e tratar outliers
- Normalizar dados, criar colunas calculadas e combinar DataFrames

## 1. Valores Ausentes (NaN)

### 🔹 Exemplo 1 — Detectando valores ausentes

In [ ]:
import pandas as pd
import numpy as np

vendas = pd.DataFrame({
    "produto": ["Caderno", "Lápis", "Mochila", "Borracha", "Caneta", "Caneta", "Estojo"],
    "categoria": ["Papelaria", "Papelaria", "Acessórios", "Papelaria", "Papelaria", "Papelaria", "Acessórios"],
    "preco": [12.5, 2.0, 150.0, np.nan, 3.5, 3.5, 9999.0],
    "quantidade_vendida": [30, 80, 5, 60, 45, 45, 2],
})

print(vendas.isna().sum())

> ⚠️ **Veja como é um erro real do Python.** Tente passar o nome de uma coluna errada para `dropna(subset=...)`: `vendas.dropna(subset=["coluna_errada"])` — você vai ver `KeyError: ['coluna_errada']`. Confira `vendas.columns` antes de informar o nome de uma coluna.

In [ ]:
try:
    vendas.dropna(subset=["coluna_errada"])
except KeyError as e:
    print(f"Erro: {e}")

### 🔹 Exemplo 2 — Tratando valores ausentes: dropna() x fillna()

In [ ]:
sem_nulos = vendas.dropna()              # remove a LINHA inteira que tem nulo
print(sem_nulos.shape)

preco_corrigido = vendas["preco"].fillna(vendas["preco"].mean())  # substitui pela média
print(preco_corrigido)

💡 **Quando usar cada um?** Use `dropna()` quando a linha sem aquele dado não serve para a análise (ex.: falta o preço, e sem preço não dá para calcular receita). Use `fillna()` quando faz sentido estimar um valor razoável no lugar do vazio (ex.: usar a média, ou 0, ou "não informado").

> ⚠️ **Armadilha real:** `vendas["preco"].fillna(0)` sozinho **não altera** `vendas` — ele só devolve uma cópia com os nulos preenchidos. Para o DataFrame original mudar de fato, reatribua: `vendas["preco"] = vendas["preco"].fillna(0)`. Esquecer isso é o erro mais comum desta semana.

### ✏️ Atividade Prática 1 — Sua vez de programar

Crie um DataFrame com pelo menos 1 valor nulo, detecte com `isna().sum()` e trate com `fillna()`, reatribuindo o resultado.

In [ ]:
meu_df = pd.DataFrame({
    "coluna_1": [...],
    "coluna_2": [...],
})

print(meu_df.isna().sum())
meu_df["coluna_2"] = meu_df["coluna_2"].fillna(0)
print(meu_df)

---
## 2. Duplicidades e Outliers

### 🔹 Exemplo 1 — Detectando e removendo duplicidades

In [ ]:
print(vendas.duplicated())          # True na 2ª ocorrência de uma linha idêntica

vendas_sem_duplicatas = vendas.drop_duplicates()
print(vendas_sem_duplicatas.shape)

> ⚠️ **Veja como é um erro real do Python.** Um erro de digitação comum é esquecer o "s" do plural: `vendas.drop_duplicate()` — você vai ver `AttributeError: 'DataFrame' object has no attribute 'drop_duplicate'`. O método certo é `drop_duplicates()`, sempre no plural.

In [ ]:
try:
    vendas.drop_duplicate()
except AttributeError as e:
    print(f"Erro: {e}")

### 🔹 Exemplo 2 — Detectando outliers com describe() e IQR

In [ ]:
print(vendas["preco"].describe())

q1 = vendas["preco"].quantile(0.25)
q3 = vendas["preco"].quantile(0.75)
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr

print(vendas[vendas["preco"] > limite_superior])

💡 **Ordem importa:** se você calcular a média (Seção 1) *antes* de tratar o outlier, a média já vem distorcida pelo valor errado — neste dataset, a média de `preco` com o outlier ainda dentro é **1695,08**; sem ele, cai para **33,6**. Trate outliers *antes* de usar a média/desvio padrão para qualquer outra decisão.

### ✏️ Atividade Prática 2 — Sua vez de programar

Crie um DataFrame com 1 linha duplicada e 1 outlier, e use `drop_duplicates()` e o método IQR para identificar os dois problemas.

In [ ]:
# duplicidades
print(meu_df.duplicated())

# outlier (IQR)
q1 = meu_df["coluna_2"].quantile(0.25)
q3 = meu_df["coluna_2"].quantile(0.75)
iqr = q3 - q1
print(meu_df[meu_df["coluna_2"] > q3 + 1.5 * iqr])

---
## 3. Normalização e Colunas Calculadas/Condicionais

### 🔹 Exemplo 1 — Normalização min-max

In [ ]:
vendas["preco_normalizado"] = (
    (vendas["preco"] - vendas["preco"].min())
    / (vendas["preco"].max() - vendas["preco"].min())
)
print(vendas[["produto", "preco", "preco_normalizado"]])

### 🔹 Exemplo 2 — Coluna calculada com transformação condicional

In [ ]:
vendas["faixa_preco"] = np.where(vendas["preco"] > 50, "Alto", "Baixo")
print(vendas[["produto", "preco", "faixa_preco"]])

💡 **Por que não um `for`?** Você poderia percorrer linha a linha com um `for` e um `if` dentro — funcionaria, mas seria mais lento e mais código. `np.where(condição, valor_se_true, valor_se_false)` faz a mesma coisa numa linha só, de forma vetorizada (mesma ideia vista na Semana 05).

### ✏️ Atividade Prática 3 — Sua vez de programar

Normalize a coluna `coluna_2` do seu DataFrame e crie uma coluna condicional com `np.where()`.

In [ ]:
meu_df["coluna_2_normalizada"] = (
    (meu_df["coluna_2"] - meu_df["coluna_2"].min())
    / (meu_df["coluna_2"].max() - meu_df["coluna_2"].min())
)
meu_df["categoria_calculada"] = np.where(meu_df["coluna_2"] > ..., "Alto", "Baixo")
print(meu_df)

---
## 4. Concatenação e Caso Real

### 🔹 Exemplo 1 — Combinando dois meses com pd.concat()

In [ ]:
vendas_fevereiro = pd.DataFrame({
    "produto": ["Caderno", "Lápis"],
    "categoria": ["Papelaria", "Papelaria"],
    "preco": [12.5, 2.0],
    "quantidade_vendida": [25, 70],
})

vendas_jan_fev = pd.concat([vendas, vendas_fevereiro], ignore_index=True)
print(vendas_jan_fev.shape)

### Caso real — O pipeline completo de limpeza

In [ ]:
vendas_csv = vendas.copy()
vendas_csv.to_csv("vendas_sujo.csv", index=False)

dados = pd.read_csv("vendas_sujo.csv")                          # 1. ler o arquivo real
dados = dados.drop_duplicates()                                 # 2. remover duplicidades
q1, q3 = dados["preco"].quantile([0.25, 0.75])
limite = q3 + 1.5 * (q3 - q1)
dados = dados[dados["preco"] <= limite]                         # 3. remover outlier (já sem distorcer a média)
dados["preco"] = dados["preco"].fillna(dados["preco"].mean())   # 4. tratar nulos (com a média já confiável)
dados["faixa_preco"] = np.where(dados["preco"] > 50, "Alto", "Baixo")  # 5. transformar

print(dados)

💡 **A ordem deste pipeline não é arbitrária:** duplicidades e outliers são removidos *antes* de calcular a média para preencher nulos — assim a média usada não vem distorcida (ver tip box da Seção 2).

### ✏️ Atividade Prática 4 — Sua vez de programar

Crie um segundo "mês" de dados e combine com `pd.concat()` antes de repetir o pipeline de limpeza.

In [ ]:
meu_df_mes2 = pd.DataFrame({
    "coluna_1": [...],
    "coluna_2": [...],
})

meus_dados_combinados = pd.concat([meu_df, meu_df_mes2], ignore_index=True)
print(meus_dados_combinados)

---
### 🏁 Fechamento — Semana 06

**Nesta semana você aprendeu:**
- Detecção e tratamento de valores ausentes: quando preencher, quando remover
- Remoção de duplicatas e tratamento de outliers com critérios claros
- Normalização, colunas calculadas e concatenação de DataFrames

**Próxima semana:** Semana 07 — Visualização e Pipelines: comunicar insights com gráficos e montar um pipeline completo.

---
## 5. Treino em Squads — Sexta-feira (Encontro 3)

Esta seção é usada **em sala (ou em salas remotas/breakout)** na sexta-feira. Cada squad recebe um dataset sujo diferente, cada um com um problema principal diferente — a missão é decidir a estratégia certa para aquele problema.

### Squad B — Cadastro de funcionários (valores ausentes no salário)

In [ ]:
dataset_squad_b = pd.DataFrame({
    "funcionario": ["Ana", "Bruno", "Carla", "Diego", "Elaine"],
    "departamento": ["Vendas", "TI", "Vendas", "RH", "TI"],
    "salario": [3200.0, None, 2800.0, None, 4500.0],
})

# explore: isna(), dropna(), fillna() — qual decisão faz mais sentido aqui?


### Squad C — Pedidos de um restaurante (linhas duplicadas)

In [ ]:
dataset_squad_c = pd.DataFrame({
    "pedido_id": [101, 102, 103, 103, 104],
    "item": ["Pizza", "Refrigerante", "Hambúrguer", "Hambúrguer", "Salada"],
    "valor": [45.0, 8.0, 32.0, 32.0, 22.0],
})

# explore: duplicated(), drop_duplicates()


### Squad D — Leituras de um sensor de temperatura (outlier)

In [ ]:
dataset_squad_d = pd.DataFrame({
    "hora": ["08h", "09h", "10h", "11h", "12h"],
    "temperatura_celsius": [22.5, 23.0, 999.0, 23.5, 24.0],
})

# explore: describe(), IQR


### Squad E — Notas de turmas em escalas diferentes (precisa normalizar)

In [ ]:
dataset_squad_e = pd.DataFrame({
    "aluno": ["Pedro", "Quenia", "Rafael", "Sofia"],
    "turma": ["A", "A", "B", "B"],
    "nota": [8.5, 7.0, 85.0, 92.0],   # turma A em escala 0-10, turma B em escala 0-100
})

# explore: normalização min-max — como comparar as notas das 2 turmas de forma justa?


### 🗣️ Debate coletivo (após as apresentações)

Depois que todos os squads apresentarem, discuta com a turma:

- Por que a mesma técnica (ex.: `dropna()`) não era a melhor escolha para todos os datasets?
- O que aconteceria se o Squad D tivesse usado `fillna()` com a média sem antes tratar o outlier?
- O que aconteceria se cada squad tivesse aplicado a solução de outro squad no seu próprio dataset?

### Desafio (opcional)

Combine os datasets dos Squads B e C com `pd.concat()` (apenas as colunas em comum) e trate os problemas de ambos no resultado combinado.

---
### Assinatura

Curso: **Análise de Dados com Python — SENAI (Turma T5)**
Semana 06 — Limpeza e Transformação de Dados

*Prof. Especialista Cláudio F. Neves*